# Physics-Informed Neural Network for Thermal Field Reconstruction

This notebook develops a Physics-Informed Neural Network (PINN) for reconstructing the temperature field of the one-dimensional thermal system.

The baseline neural network learned only from sparse, noisy, and incomplete sensor measurements.

The PINN uses the same type of neural-network architecture, but its training is additionally constrained by the governing heat equation:

$$
\frac{\partial T}{\partial t}
=
\alpha
\frac{\partial^2 T}{\partial x^2}
$$

and by the known physical boundary conditions:

$$
T(0,t)=T(1,t)=20^\circ C
$$

The goal is to investigate whether including physical knowledge improves reconstruction of the full temperature field when only limited sensor measurements are available.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

import copy

In [2]:
torch.manual_seed(42)
np.random.seed(42)

## 1. Loading the Sensor and Reference Data

The PINN is trained using the same sparse, noisy, and incomplete sensor dataset used by the baseline neural network.

The complete numerical temperature field is also loaded, but it is retained only for evaluating the final reconstruction.

It is not used as full-field training data.

In [3]:
sensor_df = pd.read_csv(
    "../data/thermal_sensor_data.csv"
)

reference_data = np.load(
    "../data/reference_temperature_field.npz"
)

x_full = reference_data["x"]
time_full = reference_data["time"]
T_true_full = reference_data["temperature_field"]

print(sensor_df.head())

print("Spatial points:", len(x_full))
print("Time points:", len(time_full))
print("Reference field shape:", T_true_full.shape)

     time_s  sensor_0.20m  sensor_0.35m  sensor_0.50m  sensor_0.65m  \
0  0.000000     20.319063     33.724598    100.323844     34.555245   
1  0.701754     19.963949     34.962053     99.511857     33.937709   
2  1.403509           NaN     34.308799     98.405535     33.585024   
3  2.105263     19.823547     34.395007     97.624852     34.447411   
4  2.807018     20.850951     35.138902     96.710022     34.539417   

   sensor_0.80m  
0     19.953629  
1     20.352297  
2     19.229874  
3     19.398539  
4     19.845936  
Spatial points: 81
Time points: 286
Reference field shape: (286, 81)


## 2. Preparing the Sensor Observations

The sensor measurements are initially stored in wide format, with one column for each sensor.

For neural-network training, they are converted into individual space-time observations of the form:

$$
(x,t,T)
$$

where:

- $x$ is the sensor position
- $t$ is the measurement time
- $T$ is the measured temperature

Measurements containing missing values are excluded from the training dataset.

In [6]:
sensor_positions = {
    "sensor_0.20m": 0.20,
    "sensor_0.35m": 0.35,
    "sensor_0.50m": 0.50,
    "sensor_0.65m": 0.65,
    "sensor_0.80m": 0.80
}

In [7]:
training_rows = []

for sensor_name, position in sensor_positions.items():

    for row_index in range(len(sensor_df)):

        time_value = sensor_df.loc[
            row_index,
            "time_s"
        ]

        temperature_value = sensor_df.loc[
            row_index,
            sensor_name
        ]

        if not pd.isna(temperature_value):

            training_rows.append(
                [
                    position,
                    time_value,
                    temperature_value
                ]
            )

In [8]:
training_df = pd.DataFrame(
    training_rows,
    columns=[
        "x",
        "time_s",
        "temperature_C"
    ]
)

print(
    "Available sensor observations:",
    len(training_df)
)

training_df.head()

Available sensor observations: 1351


,x,time_s,temperature_C
0,0.2,0.000000,20.319063
1,0.2,0.701754,19.963949
2,0.2,2.105263,19.823547
3,0.2,2.807018,20.850951
4,0.2,3.508772,20.188134


## 3. Normalizing the Sensor Data

The neural network operates more effectively when the input and output variables have comparable numerical scales.

Position, time, and temperature are therefore normalized using min-max scaling:

$$
z^*
=
\frac{z-z_{\min}}
{z_{\max}-z_{\min}}
$$

The spatial domain is known to extend from:

$$
0 \le x \le 1
$$

so the physical rod boundaries are used for the position scaling.

The same normalization strategy used for the baseline neural network is retained so that the two models can be compared consistently.

In [9]:
x_min = 0.0
x_max = 1.0

t_min = training_df["time_s"].min()
t_max = training_df["time_s"].max()

T_min = training_df["temperature_C"].min()
T_max = training_df["temperature_C"].max()

In [10]:
print("x range:", x_min, "to", x_max)
print("time range:", t_min, "to", t_max)
print("temperature range:", T_min, "to", T_max)

x range: 0.0 to 1.0
time range: 0.0 to 200.0
temperature range: 19.00723975869951 to 100.32384426905034


In [11]:
training_df["x_norm"] = (
    training_df["x"] - x_min
) / (x_max - x_min)

training_df["time_norm"] = (
    training_df["time_s"] - t_min
) / (t_max - t_min)

training_df["temperature_norm"] = (
    training_df["temperature_C"] - T_min
) / (T_max - T_min)

In [12]:
training_df.head()

,x,time_s,temperature_C,x_norm,time_norm,temperature_norm
0,0.2,0.000000,20.319063,0.2,0.000000,0.016132
1,0.2,0.701754,19.963949,0.2,0.003509,0.011765
2,0.2,2.105263,19.823547,0.2,0.010526,0.010039
3,0.2,2.807018,20.850951,0.2,0.014035,0.022673
4,0.2,3.508772,20.188134,0.2,0.017544,0.014522


## 4. Converting the Sensor Data to PyTorch Tensors

Each available sensor observation provides two neural-network inputs:

$$
(x^*,t^*)
$$

and one target:

$$
T^*
$$

The normalized values are converted to PyTorch tensors for neural-network training.

In [13]:
X_data = torch.tensor(
    training_df[
        ["x_norm", "time_norm"]
    ].values,
    dtype=torch.float32
)

y_data = torch.tensor(
    training_df[
        ["temperature_norm"]
    ].values,
    dtype=torch.float32
)

In [14]:
print("Sensor input shape:", X_data.shape)
print("Sensor target shape:", y_data.shape)

Sensor input shape: torch.Size([1351, 2])
Sensor target shape: torch.Size([1351, 1])


## 5. Heat Equation in Normalized Coordinates

The physical heat equation is:

$$
\frac{\partial T}{\partial t}
=
\alpha
\frac{\partial^2 T}{\partial x^2}
$$

However, the neural network operates using normalized position, time, and temperature:

$$
x^*=
\frac{x-x_{\min}}
{x_{\max}-x_{\min}}
$$

$$
t^*=
\frac{t-t_{\min}}
{t_{\max}-t_{\min}}
$$

$$
T^*=
\frac{T-T_{\min}}
{T_{\max}-T_{\min}}
$$

Define the scaling factors:

$$
\Delta x=x_{\max}-x_{\min}
$$

$$
\Delta t=t_{\max}-t_{\min}
$$

$$
\Delta T=T_{\max}-T_{\min}
$$

Using the chain rule:

$$
\frac{\partial T}{\partial t}
=
\frac{\Delta T}{\Delta t}
\frac{\partial T^*}{\partial t^*}
$$

and:

$$
\frac{\partial^2T}{\partial x^2}
=
\frac{\Delta T}{\Delta x^2}
\frac{\partial^2T^*}{\partial x^{*2}}
$$

Substituting these expressions into the physical heat equation gives:

$$
\frac{\partial T^*}{\partial t^*}
=
\alpha^*
\frac{\partial^2T^*}{\partial x^{*2}}
$$

where the normalized diffusivity is:

$$
\alpha^*
=
\frac{\alpha\Delta t}
{\Delta x^2}
$$

The PINN will therefore enforce the normalized physics residual:

$$
f(x^*,t^*)
=
\frac{\partial T^*}{\partial t^*}
-
\alpha^*
\frac{\partial^2T^*}{\partial x^{*2}}
$$

For a prediction that satisfies the heat equation:

$$
f(x^*,t^*)\approx0
$$

In [15]:
alpha_physical = 1.0e-4

x_scale = x_max - x_min
t_scale = t_max - t_min
T_scale = T_max - T_min

alpha_norm = (
    alpha_physical
    * t_scale
    / x_scale**2
)

print("Physical diffusivity:", alpha_physical)
print("Spatial scale:", x_scale)
print("Time scale:", t_scale)
print("Temperature scale:", T_scale)
print("Normalized diffusivity:", alpha_norm)

Physical diffusivity: 0.0001
Spatial scale: 1.0
Time scale: 200.0
Temperature scale: 81.31660451035083
Normalized diffusivity: 0.02


## 6. PINN Neural-Network Architecture

The PINN uses the same neural-network architecture as the data-driven baseline.

It receives normalized position and time:

$$
(x^*,t^*)
$$

and predicts normalized temperature:

$$
T_\theta^*(x^*,t^*)
$$

Using the same architecture helps ensure that differences between the baseline model and the PINN are mainly due to the inclusion of physical information rather than differences in model size.

In [16]:
class TemperaturePINN(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),

            nn.Linear(64, 64),
            nn.Tanh(),

            nn.Linear(64, 64),
            nn.Tanh(),

            nn.Linear(64, 1)
        )

    def forward(self, inputs):
        return self.network(inputs)

In [17]:
torch.manual_seed(42)

pinn = TemperaturePINN()

print(pinn)

TemperaturePINN(
  (network): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): Tanh()
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 7. Physics Collocation Points

The sensor measurements provide temperature information only at a small number of spatial locations.

To enforce the heat equation throughout the physical domain, additional space-time points are generated.

These points are called collocation points.

At a collocation point, no temperature measurement is supplied to the neural network.

Instead, the predicted temperature is required to approximately satisfy the governing heat equation:

$$
\frac{\partial T^*}{\partial t^*}
-
\alpha^*
\frac{\partial^2 T^*}{\partial x^{*2}}
\approx 0
$$

The collocation points therefore provide physical information without requiring additional temperature labels.

In [18]:
num_collocation = 5000

x_physics = torch.rand(
    num_collocation,
    1
)

t_physics = torch.rand(
    num_collocation,
    1
)

In [19]:
X_physics = torch.cat(
    [x_physics, t_physics],
    dim=1
)

print("Physics points shape:", X_physics.shape)

Physics points shape: torch.Size([5000, 2])


## 8. Physics Residual Using Automatic Differentiation

The PINN predicts normalized temperature as a continuous function:

$$
T_\theta^*(x^*,t^*)
$$

To determine whether this predicted function satisfies the heat equation, PyTorch automatic differentiation is used to calculate:

$$
\frac{\partial T_\theta^*}{\partial t^*}
$$

and:

$$
\frac{\partial^2 T_\theta^*}{\partial x^{*2}}
$$

The normalized heat-equation residual is then:

$$
f(x^*,t^*)
=
\frac{\partial T_\theta^*}{\partial t^*}
-
\alpha^*
\frac{\partial^2 T_\theta^*}{\partial x^{*2}}
$$

A physically consistent prediction should produce residual values close to zero.

In [20]:
def physics_residual(model, X_physics, alpha_norm):

    X = X_physics.clone().detach().requires_grad_(True)

    T_pred = model(X)

    first_derivatives = torch.autograd.grad(
        T_pred,
        X,
        grad_outputs=torch.ones_like(T_pred),
        create_graph=True
    )[0]

    dT_dx = first_derivatives[:, 0:1]
    dT_dt = first_derivatives[:, 1:2]

    second_derivatives = torch.autograd.grad(
        dT_dx,
        X,
        grad_outputs=torch.ones_like(dT_dx),
        create_graph=True
    )[0]

    d2T_dx2 = second_derivatives[:, 0:1]

    residual = (
        dT_dt
        -
        alpha_norm * d2T_dx2
    )

    return residual

In [21]:
residual_test = physics_residual(
    pinn,
    X_physics,
    alpha_norm
)

print(
    "Initial physics MSE:",
    torch.mean(residual_test**2).item()
)

NameError: name 'residual_test' is not defined